# Agent Regression Tests with Foundry Evaluators

> **DRAFT — soliciting scope feedback.** This notebook is an outline only. Code cells are intentionally empty pending review of scope, folder placement, and evaluator selection. See [README](./README.md) for context.

**Audience.** You have shipped an agent built on the OpenAI Responses API and want to catch quality regressions when you tweak the system prompt, swap models, or change a tool definition. This notebook walks through a small, end-to-end regression suite you can drop into CI.

**Tools used.**
- OpenAI Responses API with tool calling
- [`azure-ai-evaluation`](https://pypi.org/project/azure-ai-evaluation/) (open-source) for the evaluator catalog — graders are LLM-as-judge and accept any OpenAI-compatible chat model, so no Azure subscription is required to follow along.
- `pandas` for tabular comparison

**Why this notebook.** The cookbook already has [`examples/partners/macro_evals_for_agentic_systems/`](../../partners/macro_evals_for_agentic_systems/) for macro-pattern discovery from traces, and [`examples/evals/`](../../evals/) for the OpenAI Evals product. This notebook fills the gap between them: a self-contained, CI-friendly **micro-eval regression** loop driven by a fixed golden set.

## Learning objectives

By the end of this notebook you will be able to:

1. Author a small but realistic golden dataset for an agent (inputs, ground-truth answers, expected tool calls).
2. Run the agent on the dataset and capture per-turn outputs and tool-call traces.
3. Score each turn with built-in and custom Foundry evaluators using an OpenAI judge model.
4. Compare two prompt or model variants side-by-side and produce a pass/fail regression verdict.

## Open questions for reviewers

Please leave inline comments on any of these:

1. **Folder placement.** Is `examples/evaluation/agent_regression_tests_with_foundry_evaluators/` the right home, or would you prefer `examples/evals/` (the OpenAI Evals home) or `examples/partners/` (the path #2724 took)?
2. **Evaluator selection.** Current plan is the five evaluators listed below. Drop any that feel redundant, or suggest replacements.
3. **Agent scope.** Two-tool toy agent (planner + retriever) on a tiny in-memory document store, or something a touch larger?
4. **Judge model.** Use the same model the agent uses (cheap demo) or call out a stronger judge?
5. **Dataset size.** 20 items is enough to make the point and runs in well under a minute. Bigger?

## 1. Prerequisites and setup

_Install the packages from `requirements.txt`, set `OPENAI_API_KEY`, and instantiate the OpenAI client._

In [ ]:
# TODO: install deps and import openai, azure.ai.evaluation, pandas, etc.
# Example (commented):
# %pip install -q -r requirements.txt
# from openai import OpenAI
# client = OpenAI()

## 2. Build the agent under test

_A minimal two-tool agent on the Responses API: a `search_docs` retriever over a tiny in-memory corpus, and a `summarize` tool that the model can chain. Two prompt variants ("baseline" and "candidate") so we have something to compare._

In [ ]:
# TODO: define tool schemas, in-memory corpus, and run_agent(input, variant) -> AgentTrace.

## 3. Curate the golden dataset

_20 inputs covering: simple retrieval, multi-hop reasoning, ambiguous queries, refusals, and an unanswerable question. Each row carries a reference answer and the expected tool-call sequence._

In [ ]:
# TODO: load data/golden_set.jsonl into a pandas DataFrame and show the first few rows.

## 4. Run the agent on the golden set

_Iterate the dataset under each variant, capture model output, tool calls, and turn metadata into a flat results frame ready for evaluation._

In [ ]:
# TODO: run_variants(golden_df, variants=['baseline','candidate']) -> results_df

## 5. Apply the built-in evaluators

_We score every row on four built-in graders from `azure.ai.evaluation`:_

| Evaluator | What it checks |
| --- | --- |
| `GroundednessEvaluator` | Output is grounded in the retrieved context. |
| `RelevanceEvaluator` | Output answers the user's intent. |
| `IntentResolutionEvaluator` | Agent identified the right intent (especially on ambiguous queries). |
| `ToolCallAccuracyEvaluator` | Tool calls match the expected sequence and arguments. |

_All evaluators are pointed at the same OpenAI judge model for parity._

In [ ]:
# TODO: instantiate built-in evaluators with an OpenAI-compatible model config and call them per row.

## 6. Add a custom criterion

_For things the built-in catalog doesn't cover, you author a one-file evaluator. Here: an `AnswerStyleEvaluator` that fails responses which leak chain-of-thought or apologize for non-failures._

In [ ]:
# TODO: implement AnswerStyleEvaluator(__call__) returning {score, reason}.

## 7. Compare variants and gate on regression

_Aggregate per-evaluator means by variant, surface the worst-regressed rows, and emit a non-zero exit code if any criterion drops by more than a configurable threshold versus the recorded baseline._

In [ ]:
# TODO: pivot results, compute deltas, print verdict + offenders, sys.exit(non-zero on regression).

## 8. Wire it into CI

_A short walkthrough of running this notebook headless via `jupyter nbconvert --execute` from a GitHub Action, including how to store the previous run as the baseline._

## References

- [OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses)
- [`azure-ai-evaluation` documentation](https://learn.microsoft.com/python/api/overview/azure/ai-evaluation-readme)
- Cookbook: [`examples/partners/macro_evals_for_agentic_systems/`](../../partners/macro_evals_for_agentic_systems/) — the complementary macro-pattern view.
- Cookbook: [`examples/evals/`](../../evals/) — OpenAI Evals product walkthroughs.